# EduPulse — 모델 성능 비교

XGBoost, Prophet, LSTM, Ensemble 모델의 성능을 비교합니다.

**비교 항목:**
1. 개별 모델 학습 + MAPE 평가
2. 모델별 MAPE 비교 테이블
3. 예측값 vs 실제값 시각화
4. 앙상블 가중치 자동 설정
5. 분야별 모델 성능 분석

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'AppleGothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import warnings
warnings.filterwarnings('ignore')

## 1. 데이터 준비

In [ ]:
import os
data_path = '../edupulse/data/warehouse/training_dataset.csv'
if os.path.exists(data_path):
    df = pd.read_csv(data_path)
else:
    from edupulse.preprocessing.merger import build_training_dataset
    df = build_training_dataset()

print(f'학습 데이터: {df.shape}')
display(df.head())

## 2. 개별 모델 평가

In [ ]:
# XGBoost
from edupulse.model.xgboost_model import XGBoostForecaster

xgb = XGBoostForecaster()
xgb_result = xgb.evaluate(df, n_splits=5)
print(f'XGBoost MAPE: {xgb_result["mape"]:.2f}%')

In [ ]:
# Prophet
try:
    from edupulse.model.prophet_model import ProphetForecaster
    
    prophet = ProphetForecaster()
    prophet_result = prophet.evaluate(df, n_splits=5)
    print(f'Prophet MAPE: {prophet_result["mape"]:.2f}%')
except ImportError:
    print('Prophet 미설치 — pip install prophet')
    prophet_result = {'mape': float('nan')}

In [ ]:
# LSTM
try:
    from edupulse.model.lstm_model import LSTMForecaster
    
    lstm = LSTMForecaster()
    lstm_result = lstm.evaluate(df, n_splits=3)  # LSTM은 시간이 오래 걸려서 3-fold
    print(f'LSTM MAPE: {lstm_result["mape"]:.2f}%')
except ImportError:
    print('PyTorch 미설치 — pip install torch')
    lstm_result = {'mape': float('nan')}

## 3. MAPE 비교 테이블 및 시각화

In [ ]:
model_mapes = {
    'XGBoost': xgb_result['mape'],
    'Prophet': prophet_result['mape'],
    'LSTM': lstm_result['mape'],
}

comparison = pd.DataFrame({
    '모델': model_mapes.keys(),
    'MAPE (%)': [f'{v:.2f}' if not np.isnan(v) else 'N/A' for v in model_mapes.values()],
})
display(comparison)

# 시각화
valid = {k: v for k, v in model_mapes.items() if not np.isnan(v)}
if valid:
    fig, ax = plt.subplots(figsize=(8, 4))
    best = min(valid.values())
    colors = ['green' if v == best else 'steelblue' for v in valid.values()]
    ax.bar(valid.keys(), valid.values(), color=colors, alpha=0.8, width=0.5)
    ax.set_ylabel('MAPE (%)')
    ax.set_title('모델별 MAPE 비교 (낮을수록 우수)')
    for i, (name, val) in enumerate(valid.items()):
        ax.text(i, val + 0.3, f'{val:.2f}%', ha='center', fontsize=11)
    plt.tight_layout()
    plt.show()

## 4. 앙상블 모델

In [ ]:
from edupulse.model.ensemble import EnsembleForecaster

# 가용한 모델로 앙상블 구성
ensemble = EnsembleForecaster()

xgb_train = XGBoostForecaster()
xgb_train.train(df)
ensemble.add_model('xgboost', xgb_train)

try:
    prophet_train = ProphetForecaster()
    prophet_train.train(df)
    ensemble.add_model('prophet', prophet_train)
except Exception as e:
    print(f'Prophet 제외: {e}')

try:
    lstm_train = LSTMForecaster()
    lstm_train.train(df)
    ensemble.add_model('lstm', lstm_train)
except Exception as e:
    print(f'LSTM 제외: {e}')

print(f'앙상블 모델 수: {ensemble.model_count}')

In [ ]:
# 자동 가중치 설정 (MAPE 역수 기반)
weights = ensemble.auto_weight(df, n_splits=3)
print('앙상블 가중치:')
for name, w in weights.items():
    print(f'  {name}: {w:.4f}')

# 앙상블 MAPE
ensemble_result = ensemble.evaluate(df, n_splits=3)
print(f'\n앙상블 MAPE: {ensemble_result["mape"]:.2f}%')

In [ ]:
# 앙상블 포함 최종 비교
model_mapes['Ensemble'] = ensemble_result['mape']

valid = {k: v for k, v in model_mapes.items() if not np.isnan(v)}
fig, ax = plt.subplots(figsize=(8, 4))
best = min(valid.values())
colors = ['green' if v == best else ('orange' if k == 'Ensemble' else 'steelblue') for k, v in valid.items()]
ax.bar(valid.keys(), valid.values(), color=colors, alpha=0.8, width=0.5)
ax.set_ylabel('MAPE (%)')
ax.set_title('전체 모델 MAPE 비교 (앙상블 포함)')
for i, (name, val) in enumerate(valid.items()):
    ax.text(i, val + 0.3, f'{val:.2f}%', ha='center', fontsize=11)
plt.tight_layout()
plt.show()

## 5. 분야별 모델 성능 분석

In [ ]:
if 'field' in df.columns:
    fields = df['field'].unique()
    field_results = []
    
    for field in fields:
        field_df = df[df['field'] == field].sort_values('date').reset_index(drop=True)
        
        # XGBoost만 분야별 평가 (빠르므로)
        xgb_f = XGBoostForecaster()
        result = xgb_f.evaluate(field_df, n_splits=3)
        field_results.append({'분야': field, 'XGBoost MAPE (%)': f'{result["mape"]:.2f}'})
    
    field_comparison = pd.DataFrame(field_results)
    display(field_comparison)
else:
    print('field 컬럼이 없습니다.')